## Elbow load and feedback lab
Change the fingertip mass, controller load estimate and lift time. The simulated joint is torque driven. The registered pinch is a separate contact case.

In [ ]:
import sys, subprocess
from pathlib import Path

root = Path.cwd()
if not (root / "elbow_control.py").is_file():
    if (root.parent / "elbow_control.py").is_file():
        root = root.parent
    else:
        folder = root / "robot-elbow-portfolio"
        if not folder.exists():
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pyuyan0057-jpg/robot-elbow-portfolio.git", str(folder)], check=True)
        root = folder
sys.path.insert(0, str(root))
from calculate import PARAMETERS, LoadCase, solve
from elbow_control import Experiment, simulate
import matplotlib.pyplot as plt

In [ ]:
payload_g = 300
controller_estimate_g = 150
lift_s = 1.2
limit_nm = 5.0

load = LoadCase(payload_g, tuple(PARAMETERS["payload_position_mm"]), lift_s, 1.5)
_, sizing = solve(load)
print(f"Horizontal hold {sizing['hold_nm']:.3f} N m; design hold {sizing['design_hold_nm']:.3f} N m.")
print(f"Max speed {sizing['output_max_rpm']:.2f} rpm.")

In [ ]:
fig, (ax, torque) = plt.subplots(2, 1, sharex=True, layout="constrained")
for mode in ("PD", "PDG", "PIDG"):
    case = Experiment(mode, mode, payload_g, controller_estimate_g, limit_nm=limit_nm)
    rows = simulate(case, lift_s=lift_s)
    time = [r["time_s"] for r in rows]
    ax.plot(time, [r["error_deg"] for r in rows], label=mode)
    torque.plot(time, [r["applied_nm"] for r in rows], label=mode)
ax.set_ylabel("Target minus angle (deg)")
ax.legend(ncol=3)
torque.set(xlabel="Time (s)", ylabel="Applied torque (N m)")
plt.show()

### What changes when the load moves to the ball centre?
The registered case uses a specified 150 g equivalent load. The stiffness and contact tables are archived solver results for their documented cases. Changing a slider value above reruns the elbow model only.

In [ ]:
import json, csv
reg = json.loads((root / "data/registered-pinch.json").read_text())
print(f"Registered hold {reg['horizontal_hold_nm']:.3f} N m at U={reg['ball_uvw_m'][0]*1000:.1f} mm")
with (root / "data/pinch-angle-screen.csv").open() as stream:
    checks = list(csv.DictReader(stream))
for row in checks:
    print(row["angle_deg"], row["fixed_contacts_equilibrium"], f"{float(row['gravity_axis_moment_Nmm']):.4f} N mm")